# 三条 Dataset：各自扩列，需要联合推导时再汇集

```text
concepts  → 选择概念 ────────────────────────────┐
documents → 读取原文 → 清洗正文（始终同一篇文档）──┼→ 按概念按需汇集 → 联合提取／核验
images    → 检查图片字节（始终同一张图片）─────────┘
```

主线只查看概念、文档、图片三个schema。读取、清洗、检查结果扩展到原对象上；不会每走一步换一张业务表。索引、断点、来源版本、知识资产存储留在底层。

concept_ref是来源概念引用（legacy:名称/qid:QID），doc_id/image_id是对应资料的来源版本标识。长期内部概念身份迁移未完成，不按名称自动合并。

文档和图片在扩列阶段不嵌套进概念。只在身份与联合知识处理需要上下文时，按概念读取有界材料窗口。窗口结果仍需联合整合，不能把代码跑完当作知识审核通过。


## 配置与查看

默认view_saved读取实际column_pilot_v2，无新调用。MODE=run执行当前算子；处理代码／配置变化须新RUN路径。IDS=None处理全部已读概念，SAMPLE_RATE=1关闭采样，MAX_RECORDS_PER_SOURCE=None读取完整来源，SOURCE_PATHS=None使用所有支持来源。

本轮指定3个概念控制工程量；全量仍走同样的链。显示limit和sample只影响查看，不改变输入范围。


扩展列提前声明，未执行时为null。view_saved展示当前保存的列值；要观察真实扩列过程，在新运行中逐步执行即可，不将最终表误称每步之前的历史快照。

In [ ]:
from pathlib import Path
import sys, inspect, json, sqlite3, random
PROJECT=Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(PROJECT) not in sys.path:sys.path.insert(0,str(PROJECT))
from curation.v4.column_flow import ColumnFlow, BUSINESS_SCHEMAS
from curation.v4.notebook_debug import show, detail
from curation.v4.pipeline import DEFAULT
MODE='view_saved'
RUN=PROJECT/'state/curation/v4/column_pilot_v2'
DATASET=PROJECT/'datasets/demiwtg'
IDS=['legacy:木兰','legacy:芦笙','qid:Q1']
SAMPLE_RATE=1.0
MAX_RECORDS_PER_SOURCE=50_000
SOURCE_PATHS=[DATASET/x for x in ['meta/concepts.json','meta/qid_concepts.fat.jsonl.gz','meta/docs.jsonl','meta/images.jsonl','corpus/pages-en-part1.jsonl.gz']]
RAW_RUN=None  # 新运行默认读datasets；可显式复用本流程冻结的原始解析
RUN_MODEL_CELLS=False
MODEL_CONFIG={**DEFAULT,'max_calls':12}
flow=(ColumnFlow.open_saved(RUN) if MODE=='view_saved' else ColumnFlow(RUN,DATASET,SOURCE_PATHS,IDS,SAMPLE_RATE,max_records_per_source=MAX_RECORDS_PER_SOURCE,raw_run=RAW_RUN))
async def step(name):
    if MODE=='run':print(await flow.step(name))
    else:print(name,'只读保存结果')
def preview(table,limit=100,sample=False,seed=42,processed=False):
    if table not in BUSINESS_SCHEMAS:raise ValueError('主线查看concepts、documents、images')
    if limit<0:raise ValueError('limit must be nonnegative')
    condition={'concepts':'selected=1','documents':'read_status IS NOT NULL','images':'byte_status IS NOT NULL'}[table] if processed else '1=1'
    sql=f'SELECT rowid FROM {table} WHERE {condition}'
    ids=[];rng=random.Random(seed)
    if not sample:ids=[r[0] for r in flow.tables.db.execute(sql+' LIMIT ?',(limit,))]
    else:
        for i,(key,) in enumerate(flow.tables.db.execute(sql)):
            if len(ids)<limit:ids.append(key)
            else:
                j=rng.randrange(i+1)
                if j<limit:ids[j]=key
    result=[]
    for key in ids:result.extend(flow.tables.rows(table,f'SELECT * FROM {table} WHERE rowid=?',(key,)))
    return result
def schema(table):show(flow.describe(table),limit=100)
print('实际配置');detail(flow.config)


## 0．读取来源，形成三个独立 Dataset

**输入**：datasets采集概念、文档、图片及必要辅助来源。相同schema分片归入同类Dataset，原始字段和位置可追溯。

**输出**：概念身份信息进入concepts.source_details；文档和图片保留各自字段。资料已有概念引用放在concept_refs，未知／歧义等放在association_issues。这些只是来源关联，不代表语义判断。

原始额外字段、页面对应、辅助图片角色及图关系保留在底层来源存储／索引，不静默丢弃。不存在或未完整读取的来源范围可在下面查看。


In [ ]:
await step('read_sources')
concepts=flow.dataset_for('concepts')
documents=flow.dataset_for('documents')
images=flow.dataset_for('images')
print(type(documents))
with sqlite3.connect(f'file:{flow.raw_run}/records.sqlite?mode=ro',uri=True) as db:
    show([json.loads(r[0]) for r in db.execute('SELECT body FROM source_status')])

## 1．概念 Dataset：选择概念并扩列

**输入**：concept_ref、source_details及可选IDS／采样配置。

**扩展输出**：selected、selection_reason。选择在概念层进行，不按资料行抽样，不因资料多而增加入选机会。source_details内name/aliases/qid/taxonomy为来源名称、别名、外部ID及分类快照；source_id/source_path/source_row用于回查。

概念身份信息可以来自多条来源记录，但此时不把文档、图片正文放进概念行。


In [ ]:
schema('concepts')
await step('select_concepts')
show(preview('concepts',processed=True))

## 2．确定本轮资料范围（只建立关联索引，不嵌套正文）

**输入**：入选概念、文档／图片已有concept_refs及页面映射。

**处理**：分别关联文档、图片，EXISTS选择使同一资料只进入一次处理链。未知／歧义关系不作为已核验关联；未入选本轮不等于资料无效。

**输出**：可惰性遍历的documents和images Dataset；行仍分别代表一篇文档、一张图片。索引只是执行辅助，不要求逐张表调试。


In [ ]:
await step('join_documents')
await step('join_images')
documents=flow.documents()
images=flow.images()
show(preview('documents',sample=True),columns=['doc_id','title','path','concept_refs','association_issues'])
show(preview('images',sample=True),columns=['image_id','path','caption','concept_refs','association_issues'])

## 3．文档 Dataset：读取 → 保存读取进度 → 清洗 → 保存清洗进度

**输入**：一篇文档的doc_id、title、path/sections、来源定位。其他字段随记录保留。

**读取扩列**：raw_text、raw_sha256、read_status、read_error。

**清洗扩列**：clean_text、clean_status、clean_warnings、clean_blocks、clean_counts、clean_version。clean_blocks中的raw_start/raw_end/raw_text定位原文，clean_start/clean_end定位清洗正文，decision/reason记录保留排除；links/images为来源线索。位置按块对应，不冒称逐字符映射。

以下是实际调用的demiflow算子链。SaveColumns仅保存新增字段和断点；后续仍接收完整文档行。读取成功但清洗失败时，续跑复用raw_text。未知导航／模板仍可能残留，cleaned_candidate不代表可靠来源。


In [ ]:
print(inspect.getsource(ColumnFlow.document_chain))
await step('process_documents')

In [ ]:
schema('documents')
doc_sample=preview('documents',processed=True,sample=True)
show(doc_sample,columns=['doc_id','title','read_status','clean_status','clean_counts','clean_warnings'])
if doc_sample:
    d=doc_sample[0]
    detail({'doc_id':d['doc_id'],'title':d['title'],'original':d['raw_text'],'cleaned':d['clean_text']})
    show(d['clean_blocks'],limit=100)

### 单步调试文档链

也可以将上面的process_documents执行cell改成`await step('read_documents')`，先查看同一documents表新增的raw_*字段；随后执行`await step('clean_documents')`，查看继续增加的clean_*字段。不要同时重建另一套输入。完整链与单步调用使用相同算子和列缓存，成功部分会复用。


## 4．图片 Dataset：检查字节 → 扩列保存

**输入**：image_id、path、sha256、来源caption等原字段。

**扩展输出**：byte_status、byte_details（实际路径、哈希、尺寸或错误）。byte_status为空表示尚未检查。无图／解码失败保留状态，不据此判概念无材料；来源caption不代替实际看图。

此步独立于文档正文清洗，不把图片嵌套到文章或概念中，不发生图像模型调用。


In [ ]:
print(inspect.getsource(ColumnFlow.image_chain))
await step('process_images')
schema('images')
show(preview('images',processed=True),columns=['image_id','path','caption','byte_status','byte_details'])

## 5．概念 Dataset：扩展资料覆盖信息

**输入**：入选概念及独立文档、图片的处理结果。

**扩展输出**：document_count、image_count、readable_documents、verified_images、material_status、knowledge_status；只是计数与状态，不提前嵌套资料。无资料概念保留0和no_materials_in_read_scope，来源截断范围见第0步。


In [ ]:
await step('summarize_concepts')
show(preview('concepts',processed=True),columns=['concept_ref','source_details','document_count','image_count','readable_documents','verified_images','material_status','knowledge_status'])

## 6．需要联合判断时，才按概念汇集资料

身份与资料相关性判断需要“概念＋资料”；联合知识提取需要比较多份正文／图片。这时iter_knowledge_inputs才从三个Dataset按需读入有界窗口。

窗口内documents/images保留各自来源和扩展结果；旧模型接口仅在内部兼容边界转换。GROUP_SIZE为内存窗口，不是完整性标准。跨窗口语义整合仍待完善，不能把窗口列表拼接成确定知识。

下面只查看入选资料，修改CONCEPT_REF不改变采样或触发模型。


In [ ]:
CONCEPT_REF='legacy:芦笙'
show(list(flow.tables.rows('documents','SELECT d.* FROM documents d JOIN selected_document_links l USING(doc_id) WHERE l.concept_ref=? LIMIT 100',(CONCEPT_REF,))),columns=['doc_id','title','raw_text','clean_text','clean_status'])
show(list(flow.tables.rows('images','SELECT i.* FROM images i JOIN selected_image_links l USING(image_id) WHERE l.concept_ref=? LIMIT 100',(CONCEPT_REF,))),columns=['image_id','path','caption','byte_status'])

### 判定概念歧义及资料相关性 · identity

沿用现有模型算子，默认不调用模型。输入来自上一步的处理结果；identity的输入在此才按概念汇集。所有候选保留来源、版本和审核状态，未解冲突不冒称确定知识。

输入：概念source_details、文档clean_text预览、图片来源caption等元数据。输出：identity.status/target_label/reason、accepted_material_ids、rejected_materials及identity_unexamined。当前有限预览不是完整身份审核，图片元数据不等于像素核验。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('identity',MODEL_CONFIG))
else:
    print('identity: 未调用模型')

### 去重、选择并截取正文 · organize

沿用现有模型算子，默认不调用模型。输入来自上一步的处理结果；identity的输入在此才按概念汇集。所有候选保留来源、版本和审核状态，未解冲突不冒称确定知识。

输入：身份阶段接受的资料。输出：material_pack.passages（source_id、text、start/end、原文映射）、images及遗漏记录。当前去重和篇数／字符预算仍可能只取开头，不是完整知识覆盖。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('organize',MODEL_CONFIG))
else:
    print('organize: 未调用模型')

### 从多材料提取带引文的知识候选 · extract

沿用现有模型算子，默认不调用模型。输入来自上一步的处理结果；identity的输入在此才按概念汇集。所有候选保留来源、版本和审核状态，未解冲突不冒称确定知识。

输入：当前窗口的多篇正文片段。输出：facts，每项包含fact_id、statement、conditions、exceptions、evidence（source_id＋quote）；另有unresolved_conflicts和coverage_note。逐字引文匹配不等于来源可靠。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('extract',MODEL_CONFIG))
else:
    print('extract: 未调用模型')

### 对照来源修订候选与记录未解冲突 · consolidate

沿用现有模型算子，默认不调用模型。输入来自上一步的处理结果；identity的输入在此才按概念汇集。所有候选保留来源、版本和审核状态，未解冲突不冒称确定知识。

输入：同一次提取的多份原文片段及facts。输出：修订facts、changes（修改依据）、unresolved_conflicts（source_ids、issue、needed_evidence）。未解实质冲突不转成确定知识，跨窗口整合尚待实现。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('consolidate',MODEL_CONFIG))
else:
    print('consolidate: 未调用模型')

### 实际检查图片对具体知识的支持 · evidence

沿用现有模型算子，默认不调用模型。输入来自上一步的处理结果；identity的输入在此才按概念汇集。所有候选保留来源、版本和审核状态，未解冲突不冒称确定知识。

输入：具体知识候选和可读取的实际图片。输出：补充caption，以及每对image_id/fact_id的status、region、supports、limitations。只有此步实际视觉调用才判断图中区域对知识的支持。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('evidence',MODEL_CONFIG))
else:
    print('evidence: 未调用模型')

### 保存候选、来源、图片支持及阻塞 · export

沿用现有模型算子，默认不调用模型。输入来自上一步的处理结果；identity的输入在此才按概念汇集。所有候选保留来源、版本和审核状态，未解冲突不冒称确定知识。

输入：知识候选、引文、图片支持及blocked原因。输出：可追溯知识资产，保留concept_ref、task_id、来源关系和机器候选状态；无资料／身份歧义等阻塞继续保留。资料扩列状态不代表知识审核完成。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('export',MODEL_CONFIG))
else:
    print('export: 未调用模型')

## 运行结果与后续边界

资料扩列不等于知识提取完成。知识资产仍独立保存陈述、条件、引文、图片支持与审核状态；它们是最终业务输出，不是为清洗每一步另建中间表。当前实际运行止于三个Dataset的处理与覆盖扩列。


In [ ]:
show([{'dataset':name,'rows':flow.tables.db.execute(f'SELECT count(*) FROM {name}').fetchone()[0]} for name in BUSINESS_SCHEMAS])
print('模型请求数',len(list(RUN.rglob('request.json'))))